# DAAAnalyzer_0.0.AnalyzeDPOAEs.ipynb
Created by: JFranco | Created On: 24 DEC 2024 | Last Env: QISHA | Last Run Date: 25 DEC 2024

This Python notebook initializes the analysis for a specific DAA Run. It will create repository folders, if they do not already exist,
and construct a df for storing threshold information that the user can open outside of Python to manually enter ABR thresholds.

In [2]:
import numpy as np
import pandas as pd
import glob
import os
import xlrd

In [3]:
#                *** WHAT TO ANALYZE // WHERE TO GET/STORE **
# Key identifiers
prepID = 'CMI_005'
daaRun = '3'
daaType = '7D Post Noise Exposure'

# Directories 
#   - Existing ones - 
dirMain = '/Users/joyfranco/Dropbox (Partners HealthCare)/JF_Shared/Data/CFTS/'
dirPrep = dirMain+prepID+'/'
dirData = dirPrep+'RawData/'
dirMD = dirPrep+'Metadata/'

#   - Ones that need to be made -
dirSV = dirPrep+'Results/'
dirDFs = dirSV+"1_CompiledMeasurements/"
dirDPOAEs = dirSV+"2_DPOAEPlotsByGroup/"
dirSjABRPlts = dirSV+"3_ABRPlotsBySubject/"
dirGpABRPlts = dirSV+"4_ABRPlotsByGroup/"
dirSjWOPlts = dirSV+"5_Wave1PlotsBySubject/"
dirGpWOPlts = dirSV+"6_Wave1PlotsByGroup/"

# Filenames
#    - Existing ones -
fnMD = prepID+'.Metadata.AnimalInfo.csv'
fnIso = 'IsoDP-'+daaRun+'-1.tsv'

#    - Ones to make -
fnDPs = prepID+'.DPOAEf2Thresholds.DAA_'+daaRun+'.csv'
fnThs = prepID+'.ABRThresholds.DAA_'+daaRun+'.csv'

In [4]:
#      *** MAKE SUBFOLDERS AS NECESSARY**
# Create directory for storing spreadsheets and summary plots if they don't exist
if not os.path.exists(dirSV): os.mkdir(dirSV)
if not os.path.exists(dirDFs): os.mkdir(dirDFs)
if not os.path.exists(dirDPOAEs): os.mkdir(dirDPOAEs)
if not os.path.exists(dirSjABRPlts): os.mkdir(dirSjABRPlts)
if not os.path.exists(dirGpWOPlts): os.mkdir(dirGpABRPlts)
if not os.path.exists(dirSjWOPlts): os.mkdir(dirSjWOPlts)
if not os.path.exists(dirGpWOPlts): os.mkdir(dirGpWOPlts)

In [5]:
#    *** LOAD MD SHEETS **
# Sample metadata sheet that include animal and condition info
dfMD = pd.read_csv(dirMD+fnMD)
dfMD.reset_index(inplace=True)
dfMD

,index,AnimalID,Sex,DOB,CMIID,InjectionDate,AgeAtInjection,Package,InjGroup,NE,NEDate,AgeAtNE,NELevel,NEBand,NEDuration,NEGroup
0,0,C032,F,10/8/24,CMI_005,11/25/24,7,AAV,CtrlInj,NaN,NaN,NaN,97.5,8_16,2h,NaN
1,1,C033,M,10/8/24,CMI_005,11/25/24,7,AAV,CtrlInj,NaN,NaN,NaN,97.5,8_16,2h,NaN
2,2,C034,F,10/8/24,CMI_005,11/25/24,7,AAV,CtrlInj,NaN,NaN,NaN,97.5,8_16,2h,NaN
3,3,C035,M,10/8/24,CMI_005,11/25/24,7,AAV,HABclw,NaN,NaN,NaN,97.5,8_16,2h,NaN
4,4,C036,F,10/8/24,CMI_005,11/25/24,7,AAV,HABclw,NaN,NaN,NaN,97.5,8_16,2h,NaN
5,5,C037,M,10/8/24,CMI_005,11/25/24,7,AAV,HABclw,NaN,NaN,NaN,97.5,8_16,2h,NaN
6,6,C038,F,10/8/24,CMI_005,11/25/24,7,AAV,UnInj,NaN,NaN,NaN,97.5,8_16,2h,NaN
7,7,C039,M,10/8/24,CMI_005,11/25/24,7,AAV,UnInj,NaN,NaN,NaN,97.5,8_16,2h,NaN


In [6]:
# *** GENERATE A LIST OF DAA FOLDERS THAT WILL BE ANALYZED **
# Get a list of all the folders that match the prep
allFolders = os.listdir(dirData)
folderList = []
for folder in allFolders:
    if ('.'+daaRun) in folder:
        folderList.append(folder)
folderList.sort()
folderList

['CMI_005.C032.3',
 'CMI_005.C033.3',
 'CMI_005.C034.3',
 'CMI_005.C035.3',
 'CMI_005.C036.3',
 'CMI_005.C037.3',
 'CMI_005.C038.3',
 'CMI_005.C039.3']

In [7]:
# *** READ IN EACH ISOLINE TSV FILE FOR EACH DPOAE**
dfDPs = pd.DataFrame()
for folder in folderList:
    # Setup the path to the file
    path = dirData+folder+'/'+fnIso
    # Read in the file & make adjustments for readability
    dfIso = pd.read_csv(path,  sep='\t')
    dfIso['Vals'] = dfIso.index
    # Iterate through all of the rows available and get values 
    freqs = []
    threshs = []
    for index, row in dfIso.iterrows():
        f2Freq = row['Vals'][0]
        f2Freq = int(float(f2Freq)/1000)
        if (f2Freq != 64):
            freqs.append(f2Freq)
            thresh = row['Vals'][3]
            if thresh =='   NaN':
                thresh = 75
            threshs.append(int(float(thresh)))
    data = {'f2Freq_kHz': freqs, 'Thresh_dBSPL': threshs}
    df = pd.DataFrame(data)
    anID = folder[8:-2]
    df['AnimalID'] = anID
    df['CMIID'] = prepID
    df['DAA_Run'] = daaRun
    # Get information about this animal from the MD sheet
    inMD = dfMD.index[dfMD['AnimalID']==anID].tolist()[0]
    inj = dfMD.loc[inMD]['InjGroup']
    sex = dfMD.loc[inMD]['Sex']
    df['InjGroup'] = inj
    df['Sex'] = sex
    dfDPs = pd.concat([dfDPs,df])
# Reset index and save DPOAE thresholds as CSV
dfDPs.reset_index(inplace=True)
dfDPs.drop(['index'], axis=1, inplace=True)
dfDPs.to_csv(dirDFs+fnDPs) 

In [8]:
# *** GENERATE DATAFRAME FOR ALL THRESHOLD DATA ***
dfThs = pd.DataFrame()
for folder in folderList:
    # Setup the path to the file
    abrFold = dirData+folder+'/'
    # Get a list of available ABR files
    allFiles = os.listdir(abrFold)
    allFiles.sort()
    for file in allFiles:
        # Get the file extension
        fname, fext = os.path.splitext(file)
        # Only consider files that are analyzed 
        if fext=='.tsv':
            if 'ABR' in fname:
                # Get information about the recording subject
                anID = folder[8:-2]
                # Get information about this animal from the MD sheet
                inMD = dfMD.index[dfMD['AnimalID']==anID].tolist()[0]
                inj = dfMD.loc[inMD]['InjGroup']
                sex = dfMD.loc[inMD]['Sex']
                # Get information about the frequency from this file
                dfFile = pd.read_csv(abrFold+file, lineterminator='\n',encoding='unicode_escape')
                fInfo = dfFile.columns[0]
                inPre = fInfo.index("SW FREQ: ")
                inPost = fInfo.index("\t# AVERAGES")
                freq = fInfo[(inPre+9):(inPost-3)]
                # Compile information into df
                data = {'AnimalID':anID,"CMIID":prepID,"Sex":sex,"InjGroup":inj, "File":file,"DAA_Run":daaRun,
                        "Freq_kHz":freq,"Th_dB_Unblinded":''}
                df = pd.DataFrame(data, index=[0])
            
                # Add info to df for all 
                dfThs = pd.concat([dfThs,df])
  

# Save the main df that has ABR data for all files in this run 
dfThs.reset_index(inplace=True)
dfThs.drop(['index'], axis=1, inplace=True)
dfThs.to_csv(dirDFs+fnThs) 